In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

ACCOUNT_TABLE = "dbx_fintech_data_platform.silver.accounts"

CDC_TABLE = "dbx_fintech_data_platform.bronze.account_cdc"

SCD2_TABLE = "dbx_fintech_data_platform.silver.accounts_scd2"

In [0]:
%sql

CREATE TABLE IF NOT EXISTS dbx_fintech_data_platform.silver.accounts_scd2 (
    account_id STRING,
    customer_id STRING,
    account_type STRING,
    currency STRING,
    status STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    effective_from TIMESTAMP,
    effective_to TIMESTAMP,
    is_current BOOLEAN
)
USING DELTA;

In [0]:
scd2_count = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM {SCD2_TABLE}
""").collect()[0]["cnt"]

print(f"Existing SCD2 records: {scd2_count}")

In [0]:
if scd2_count == 0:

    spark.sql(f"""
        INSERT INTO {SCD2_TABLE}
        SELECT
            account_id,
            customer_id,
            account_type,
            currency,
            status,
            created_at,
            updated_at,

            COALESCE(
                updated_at,
                created_at,
                current_timestamp()
            ) AS effective_from,

            CAST(NULL AS TIMESTAMP) AS effective_to,

            TRUE AS is_current

        FROM {ACCOUNT_TABLE}
    """)

    print("Initial SCD2 load completed.")

else:

    print("SCD2 table already initialized. Skipping initial load.")

In [0]:
%sql

SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT account_id) AS unique_accounts,
    SUM(CASE WHEN is_current = TRUE THEN 1 ELSE 0 END) AS current_records
FROM dbx_fintech_data_platform.silver.accounts_scd2;

In [0]:
cdc_df = spark.table(CDC_TABLE)

print("CDC records:", cdc_df.count())

display(
    cdc_df.orderBy("change_timestamp").limit(20)
)

In [0]:
effective_cdc_df = (
    cdc_df
    .filter(F.col("change_type") == "UPDATE")
    .filter(
        F.coalesce(F.col("old_value"), F.lit("")) !=
        F.coalesce(F.col("new_value"), F.lit(""))
    )
)

In [0]:
print(
    "Effective CDC changes:",
    effective_cdc_df.count()
)

In [0]:
window_spec = (
    Window
    .partitionBy("account_id", "changed_column")
    .orderBy(F.col("change_timestamp").desc())
)

latest_cdc_df = (
    effective_cdc_df
    .withColumn("_rn", F.row_number().over(window_spec))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

In [0]:
pivoted_cdc_df = (
    latest_cdc_df
    .groupBy("account_id")
    .pivot(
        "changed_column",
        ["account_type", "status"]
    )
    .agg(F.first("new_value"))
)

In [0]:
latest_timestamp_df = (
    latest_cdc_df
    .groupBy("account_id")
    .agg(
        F.max("change_timestamp").alias("cdc_change_timestamp")
    )
)

cdc_ready_df = (
    pivoted_cdc_df
    .join(
        latest_timestamp_df,
        on="account_id",
        how="inner"
    )
)

display(cdc_ready_df)

In [0]:
cdc_ready_df.createOrReplaceTempView(
    "account_cdc_scd2_ready"
)

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW account_scd2_changes AS

SELECT
    t.account_id,

    t.customer_id,

    CASE
        WHEN s.account_type IS NOT NULL
        THEN s.account_type
        ELSE t.account_type
    END AS account_type,

    t.currency,

    CASE
        WHEN s.status IS NOT NULL
        THEN s.status
        ELSE t.status
    END AS status,

    t.created_at,

    s.cdc_change_timestamp AS change_timestamp

FROM dbx_fintech_data_platform.silver.accounts_scd2 t

INNER JOIN account_cdc_scd2_ready s
    ON t.account_id = s.account_id

WHERE t.is_current = TRUE;

In [0]:
%sql

MERGE INTO dbx_fintech_data_platform.silver.accounts_scd2 AS target

USING account_scd2_changes AS source

ON target.account_id = source.account_id
AND target.is_current = TRUE

WHEN MATCHED THEN
UPDATE SET
    target.effective_to = source.change_timestamp,
    target.is_current = FALSE;

In [0]:
%sql

INSERT INTO dbx_fintech_data_platform.silver.accounts_scd2
(
    account_id,
    customer_id,
    account_type,
    currency,
    status,
    created_at,
    updated_at,
    effective_from,
    effective_to,
    is_current
)

SELECT
    account_id,
    customer_id,
    account_type,
    currency,
    status,
    created_at,
    change_timestamp AS updated_at,
    change_timestamp AS effective_from,
    CAST(NULL AS TIMESTAMP) AS effective_to,
    TRUE AS is_current

FROM account_scd2_changes;

In [0]:
%sql

SELECT
    account_id,
    account_type,
    status,
    effective_from,
    effective_to,
    is_current
FROM dbx_fintech_data_platform.silver.accounts_scd2
ORDER BY account_id, effective_from
LIMIT 50;

In [0]:
%sql

SELECT
    account_id,
    COUNT(*) AS current_records
FROM dbx_fintech_data_platform.silver.accounts_scd2
WHERE is_current = TRUE
GROUP BY account_id
HAVING COUNT(*) != 1;

In [0]:
%sql

SELECT
    COUNT(*) AS total_versions,
    COUNT(DISTINCT account_id) AS accounts,
    SUM(
        CASE
            WHEN is_current = TRUE THEN 1
            ELSE 0
        END
    ) AS current_versions,
    SUM(
        CASE
            WHEN is_current = FALSE THEN 1
            ELSE 0
        END
    ) AS historical_versions
FROM dbx_fintech_data_platform.silver.accounts_scd2;

In [0]:
%sql

SELECT
    account_id,
    COUNT(*) AS current_count
FROM dbx_fintech_data_platform.silver.accounts_scd2
WHERE is_current = TRUE
GROUP BY account_id
HAVING COUNT(*) > 1;

In [0]:
%sql

SELECT *
FROM dbx_fintech_data_platform.silver.accounts_scd2
WHERE account_id = (
    SELECT account_id
    FROM account_scd2_changes
    LIMIT 1
)
ORDER BY effective_from;

In [0]:
%sql

SELECT *
FROM dbx_fintech_data_platform.silver.accounts_scd2
WHERE is_current = TRUE;

In [0]:
%sql

SELECT
    account_id,
    account_type,
    status,
    effective_from,
    effective_to,
    is_current
FROM dbx_fintech_data_platform.silver.accounts_scd2
WHERE account_id = (
    SELECT account_id
    FROM account_scd2_changes
    LIMIT 1
)
ORDER BY effective_from;